In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

np.random.seed(42)

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, 2000).clip(5, 5000),
    'hour': np.random.normal(14, 4, 2000).clip(0, 23).astype(int),
    'is_electronics': np.random.binomial(1, 0.3, 2000),
    'tx_per_day': np.random.poisson(3, 2000),
    'fraud': 0
})

# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, 100),
    'hour': np.random.choice([0,1,2,3,4,5,22,23], 100),
    'is_electronics': np.random.binomial(1, 0.7, 100),
    'tx_per_day': np.random.poisson(8, 100),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [6]:
features = ['amount', 'hour', 'is_electronics', 'tx_per_day']
X = df[features]
y = df['fraud']

# TWÓJ KOD
# 1. train_test_split (80/20, stratify=y)
# 2. RandomForestClassifier(100)
# 3. classification_report
# 4. pickle.dump do 'fraud_model.pkl'

# 1
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 3
model.fit(X_train, y_train)

# 4
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# 5. zapis modelu
with open("fraud_model.pkl", "wb") as f:
    pickle.dump(model, f)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420



In [9]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    hour: int
    is_electronics: int
    tx_per_day: int

# TWÓJ KOD
# Endpoint POST /score
# Przyjmij Transaction, zwróć: {"is_fraud": bool, "fraud_probability": float}

@app.post("/score")
def score(tx: Transaction):
    data = np.array([[tx.amount, tx.hour, tx.is_electronics, tx.tx_per_day]])
    
    prob = model.predict_proba(data)[0][1]
    pred = model.predict(data)[0]
    
    return {
        "is_fraud": bool(pred),
        "fraud_probability": float(prob)
    }
    
@app.get("/health")
def health():
    return {"status": "ok"}

Overwriting fraud_api.py


In [8]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "hour": 14, "is_electronics": 0, "tx_per_day": 3})
print("Normalna:", r.json())

# TWÓJ KOD — przetestuj podejrzaną transakcję (amount=5500, hour=3, is_electronics=1, tx_per_day=12)

r2 = requests.post(
    "http://localhost:8001/score",
    json={"amount": 5500, "hour": 3, "is_electronics": 1, "tx_per_day": 12}
)
print("Podejrzana:", r2.json())


Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 1.0}


In [5]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('transactions', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

# TWÓJ KOD
# Dla każdej transakcji:
# 1. Wyciągnij cechy (amount, hour z timestamp, is_electronics, tx_per_day=5)
# 2. requests.post(API_URL, json=features)
# 3. Jeśli is_fraud: wyślij do tematu 'alerts', wypisz ALERT

for message in consumer:
    tx = message.value

    # 1
    features = {
        "amount": tx["amount"],
        "hour": tx.get("hour", 12),
        "is_electronics": 1 if tx["category"] == "elektronika" else 0,
        "tx_per_day": 5
    }

    # 2
    response = requests.post(API_URL, json=features)
    result = response.json()

    # 3
    if result["is_fraud"]:
        alert_producer.send("alerts", value={
            "tx": tx,
            "score": result["fraud_probability"]
        })

        print(f"ALERT ML: {tx['tx_id']} | prob={result['fraud_probability']}")

Writing ml_consumer.py


In [ ]:
#Praca domowa
#1. Porównanie: reguły vs ML
    #W projekcie zastosowano dwa podejścia do wykrywania fraudów:
    #1. Scoring regułowy (Ćw. 1)
    #   Opiera się na prostych zasadach, np. wysoka kwota, nietypowa godzina, kategoria produktu.
    #   Jest łatwy do interpretacji, ale ma ograniczoną skuteczność — wykrywa tylko znane wzorce.
    #2. Model ML (Random Forest)
    #   Uczy się na danych i potrafi wykrywać bardziej złożone zależności między cechami.
    #   Dzięki temu lepiej identyfikuje nietypowe przypadki fraudów.
    #Wynik:
    #Model ML wykrywa więcej przypadków i jest bardziej elastyczny, natomiast podejście regułowe jest prostsze i bardziej przejrzyste.
